## Ноутбук 02: EDA и предобработка отзывов пользователей (Reviews)
### Цели и задачи исследования:
1. **Анализ разреженности (Sparsity):** Исследование распределения количества отзывов на одного пользователя и на одну игру.
2. **Фильтрация неактивных сущностей (Cold Start):** Отсечение «холодных» пользователей с единичными отзывами для стабильности алгоритмов факторизации.
3. **Обработка целевой переменной:** Анализ времени игры (`playtime_forever`) и его логарифмирование (`log1p`).
4. **Хронологический сплит (Train/Test Split):** Разбиение взаимодействий по времени (`Leave-Last-1-Out`) без дата-лика.

In [1]:
import polars as pl
import os
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

data_dir = '../data'
output_dir = '../data/processed'

### Загрузка датасетов

In [2]:
reviews_df = pl.read_csv(
    os.path.join(data_dir, 'reviews.csv'),
    ignore_errors=True
)

reviews_df.head(3)

recommendationid,appid,author_steamid,author_num_games_owned,author_num_reviews,author_playtime_forever,author_playtime_last_two_weeks,author_playtime_at_review,author_last_played,language,review_text,timestamp_created,timestamp_updated,voted_up,votes_up,votes_funny,weighted_vote_score,comment_count,steam_purchase,received_for_free,written_during_early_access,created_at,updated_at
i64,i64,i64,i64,i64,i64,i64,f64,i64,str,str,i64,i64,bool,i64,i64,f64,i64,bool,bool,bool,str,str
10000000,264220,76561198085405844,760,74,12,0,12.0,1399059573,"""polish""","""What's a crap. This game costs…",1399059965,1399059965,true,0,1,0.459906,0,true,false,false,"""2025-09-07 12:51:00.564782+00:…","""2025-09-08 00:47:55.754043+00:…"
100001066,1006440,76561198014439859,485,234,424,0,424.0,1632666417,"""russian""","""Игра в жанре квеста point-&-cl…",1632673992,1632673992,true,8,0,0.63077,0,true,false,false,"""2025-09-07 12:51:00.564782+00:…","""2025-09-08 00:47:55.754043+00:…"
100002344,320721,76561198048038590,0,385,0,0,null,0,"""german""","""Erneut gibt es einen DLC mit d…",1632675631,1632675631,false,1,0,0.52381,0,true,false,false,"""2025-09-07 12:51:00.564782+00:…","""2025-09-08 00:47:55.754043+00:…"


In [3]:
reviews_df.glimpse()

Rows: 1048148
Columns: 23
$ recommendationid                <i64> 10000000, 100001066, 100002344, 100002361, 100002504, 100002591, 100003580, 100003676, 100003804, 100005000
$ appid                           <i64> 264220, 1006440, 320721, 1604700, 1338560, 1418320, 1232500, 726870, 1721670, 1638870
$ author_steamid                  <i64> 76561198085405844, 76561198014439859, 76561198048038590, 76561197994386273, 76561198138996331, 76561199208288640, 76561199089236340, 76561198155272674, 76561198056175175, 76561199059802534
$ author_num_games_owned          <i64> 760, 485, 0, 0, 816, 0, 0, 0, 315, 0
$ author_num_reviews              <i64> 74, 234, 385, 3, 26, 1, 2, 13, 14, 4
$ author_playtime_forever         <i64> 12, 424, 0, 86, 25, 60, 311, 639, 0, 12
$ author_playtime_last_two_weeks  <i64> 0, 0, 0, 0, 0, 0, 0, 0, 0, 0
$ author_playtime_at_review       <f64> 12.0, 424.0, null, 86.0, 25.0, 60.0, 261.0, 188.0, null, 12.0
$ author_last_played              <i64> 1399059573, 1632666417, 0,